# 🎮 Hybrid Chess — Game Rules & Environment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lessen-xu/Hybrid-Chess/blob/main/notebooks/01_game_rules_and_env.ipynb)

This notebook introduces the Hybrid Chess game and its Python environment API.

**What you'll learn:**
1. What Hybrid Chess is — International Chess vs Chinese Chess on a shared board
2. How to create game environments and make moves
3. How to visualize the board state
4. How to create custom rule variants
5. How to use the Gymnasium interface

---

## 0. Setup

If running on **Google Colab**, uncomment and run the cell below to install the package:

In [ ]:
# Uncomment for Google Colab:
# !pip install -q git+https://github.com/lessen-xu/Hybrid-Chess.git

In [ ]:
# Verify installation
import hybrid
print(f"Hybrid Chess loaded from: {hybrid.__file__}")

---

## 1. What is Hybrid Chess?

Hybrid Chess is an **asymmetric board game** where one side plays with International Chess pieces and the other with Chinese Chess (Xiangqi) pieces — on the same 9×10 board.

```
     Xiangqi Side (top, y=6–9)
  ┌──────────────────────────┐
  │  c  h  e  a  g  a  e  h  c  │  y=9  (back rank)
  │  .  n  .  .  .  .  .  n  .  │  y=7  (cannons)
  │  s  .  s  .  s  .  s  .  s  │  y=6  (soldiers)
  │  ─ ─ ─ ─ RIVER ─ ─ ─ ─ ─  │  y=5
  │  .  .  .  .  .  .  .  .  .  │  y=4
  │  .  .  .  .  .  .  .  .  .  │  y=3
  │  .  .  .  .  .  .  .  .  .  │  y=2
  │  P  P  P  P  P  P  P  P  P  │  y=1  (pawns)
  │  R  N  B  Q  K  B  N  R  .  │  y=0  (back rank)
  └──────────────────────────┘
     Chess Side (bottom, y=0–1)

  Uppercase = Chess pieces (K Q R B N P)
  Lowercase = Xiangqi pieces (g a e h c n s)
```

### Key Differences from Standard Chess
- **Xiangqi rules apply to Xiangqi pieces**: Horses have leg-blocking, Elephants can't cross the river, General is confined to the palace, Cannons jump to capture
- **Flying General**: If the General and King face each other on the same column with nothing between, the General can capture the King
- **Stalemate = loss** (Xiangqi convention, not a draw)
- **9 pawns** for Chess (9 columns), 5 soldiers for Xiangqi

---

## 2. Creating the Environment

In [ ]:
from hybrid.core.env import HybridChessEnv
from hybrid.core.render import render_board
from hybrid.core.types import Side

# Create a standard game environment
env = HybridChessEnv()

# Reset to initial position
state = env.reset()

print(f"Side to move: {state.side_to_move.name}")
print(f"Ply (half-moves): {state.ply}")
print()
print(render_board(state.board))

### Reading the Board

| Symbol | Piece | Side |
|--------|-------|------|
| `K` | King | Chess |
| `Q` | Queen | Chess |
| `R` | Rook | Chess |
| `B` | Bishop | Chess |
| `N` | Knight | Chess |
| `P` | Pawn | Chess |
| `g` | General (將) | Xiangqi |
| `a` | Advisor (士) | Xiangqi |
| `e` | Elephant (象) | Xiangqi |
| `h` | Horse (馬) | Xiangqi |
| `c` | Chariot (車) | Xiangqi |
| `n` | Cannon (砲) | Xiangqi |
| `s` | Soldier (卒) | Xiangqi |

---

## 3. Making Moves

In [ ]:
from hybrid.core.types import Move

# Get all legal moves for the current side (Chess)
legal_moves = env.legal_moves()
print(f"{len(legal_moves)} legal moves for {state.side_to_move.name}")
print()

# Show first 10 moves
for mv in legal_moves[:10]:
    piece = state.board.get(mv.fx, mv.fy)
    print(f"  {piece.kind.name:8s} ({mv.fx},{mv.fy}) → ({mv.tx},{mv.ty})")
print(f"  ... and {len(legal_moves) - 10} more")

In [ ]:
# Let's play a few moves!
import random
random.seed(42)

env = HybridChessEnv()
state = env.reset()

print("=" * 40)
print("Initial Position")
print("=" * 40)
print(render_board(state.board))

for i in range(6):
    legal = env.legal_moves()
    move = random.choice(legal)
    piece = state.board.get(move.fx, move.fy)
    
    state, reward, done, info = env.step(move)
    
    print(f"\n--- Move {i+1}: {piece.side.name}'s {piece.kind.name} "
          f"({move.fx},{move.fy}) → ({move.tx},{move.ty}) ---")
    print(f"Reward: {reward}, Done: {done}")
    print(render_board(state.board))
    
    if done:
        print(f"\nGame over! Reason: {info.reason}")
        break

---

## 4. Playing a Complete Random Game

Let's simulate a full game with two random players to see how the environment handles termination.

In [ ]:
def play_random_game(seed=0, verbose=False):
    """Play a complete random game and return statistics."""
    rng = random.Random(seed)
    env = HybridChessEnv()
    state = env.reset()
    
    move_count = 0
    while True:
        legal = env.legal_moves()
        if not legal:
            break
        move = rng.choice(legal)
        state, reward, done, info = env.step(move)
        move_count += 1
        
        if done:
            break
    
    return {
        "moves": move_count,
        "result": info.status if info else "unknown",
        "reason": info.reason if info else "unknown",
        "winner": info.winner.name if (info and info.winner) else "draw",
    }

# Play 20 random games
results = [play_random_game(seed=i) for i in range(20)]

print(f"{'Game':>5} {'Moves':>6} {'Winner':>10} {'Reason'}")
print("-" * 55)
for i, r in enumerate(results):
    print(f"{i+1:>5} {r['moves']:>6} {r['winner']:>10} {r['reason']}")

# Summary statistics
avg_moves = sum(r["moves"] for r in results) / len(results)
chess_wins = sum(1 for r in results if r["winner"] == "CHESS")
xiangqi_wins = sum(1 for r in results if r["winner"] == "XIANGQI")
draws = sum(1 for r in results if r["winner"] == "draw")

print(f"\n📊 Summary of {len(results)} random games:")
print(f"   Chess wins:   {chess_wins} ({chess_wins/len(results)*100:.0f}%)")
print(f"   Xiangqi wins: {xiangqi_wins} ({xiangqi_wins/len(results)*100:.0f}%)")
print(f"   Draws:        {draws} ({draws/len(results)*100:.0f}%)")
print(f"   Avg moves:    {avg_moves:.0f}")

---

## 5. Custom Rule Variants

One of the most powerful features is `VariantConfig` — you can mix and match rule tweaks without touching source code.

In [ ]:
from hybrid.core.config import VariantConfig

# Standard game
standard = HybridChessEnv()
s = standard.reset()

# No queen variant — removes Chess's strongest piece
no_queen = HybridChessEnv(variant=VariantConfig(no_queen=True))
s_nq = no_queen.reset()

# Extra cannon — gives Xiangqi a 3rd cannon
extra_cannon = HybridChessEnv(variant=VariantConfig(extra_cannon=True))
s_ec = extra_cannon.reset()

# Combined: no queen + extra cannon (balanced?)
balanced = HybridChessEnv(variant=VariantConfig(no_queen=True, extra_cannon=True))
s_bal = balanced.reset()

print("=== Standard ===")
print(render_board(s.board))
print(f"Legal moves: {len(standard.legal_moves())}")

print("\n=== No Queen ===")
print(render_board(s_nq.board))
print(f"Legal moves: {len(no_queen.legal_moves())}")

print("\n=== Extra Cannon ===")
print(render_board(s_ec.board))
print(f"Legal moves: {len(extra_cannon.legal_moves())}")

print("\n=== No Queen + Extra Cannon ===")
print(render_board(s_bal.board))
print(f"Legal moves: {len(balanced.legal_moves())}")

In [ ]:
# All available variant options:
print("Available VariantConfig options:")
print()
for field_name, field_obj in VariantConfig.__dataclass_fields__.items():
    default = field_obj.default
    print(f"  {field_name:25s} default={default}")

---

## 6. FEN Notation — Load Any Position

You can set up any board position using a FEN-like notation string.

In [ ]:
from hybrid.core.fen import parse_fen, board_to_fen

# Create an endgame position
endgame_fen = "4g4/9/9/9/9/9/9/9/4P4/4K4 c"
board, side = parse_fen(endgame_fen)

print(f"FEN: {endgame_fen}")
print(f"Side to move: {side.name}")
print()
print(render_board(board))

# Load it into an environment
env = HybridChessEnv()
state = env.reset_from_fen(endgame_fen)
print(f"\n{len(env.legal_moves())} legal moves from this position")

In [ ]:
# Round-trip: board → FEN → board
env = HybridChessEnv()
state = env.reset()

fen = board_to_fen(state.board, state.side_to_move)
print(f"Standard starting FEN:\n{fen}")

# Verify round-trip
board2, side2 = parse_fen(fen)
fen2 = board_to_fen(board2, side2)
assert fen == fen2, "Round-trip failed!"
print("\n✅ FEN round-trip verified!")

---

## 7. Using AI Agents

In [ ]:
from hybrid.agents.random_agent import RandomAgent
from hybrid.agents.greedy_agent import GreedyAgent

# Create agents
random_agent = RandomAgent()
greedy_agent = GreedyAgent()

# Random vs Greedy — 10 games
results = {"random_wins": 0, "greedy_wins": 0, "draws": 0}

for game_i in range(10):
    env = HybridChessEnv()
    state = env.reset()
    agents = {Side.CHESS: random_agent, Side.XIANGQI: greedy_agent}
    
    while True:
        legal = env.legal_moves()
        if not legal:
            break
        agent = agents[state.side_to_move]
        move = agent.select_move(state, legal)
        state, reward, done, info = env.step(move)
        if done:
            break
    
    if info and info.winner == Side.CHESS:
        results["random_wins"] += 1
    elif info and info.winner == Side.XIANGQI:
        results["greedy_wins"] += 1
    else:
        results["draws"] += 1

print("Random (Chess) vs Greedy (Xiangqi) — 10 games:")
print(f"  Random wins:  {results['random_wins']}")
print(f"  Greedy wins:  {results['greedy_wins']}")
print(f"  Draws:        {results['draws']}")

---

## 8. Gymnasium Interface

For compatibility with standard RL frameworks (Stable-Baselines3, CleanRL, etc.):

In [ ]:
try:
    import gymnasium as gym
    import hybrid.gym_env  # registers HybridChess-v0
    
    env = gym.make("HybridChess-v0")
    obs, info = env.reset()
    
    print(f"Observation shape: {obs.shape}  (14 channels × 10 rows × 9 cols)")
    print(f"Action space: {env.action_space}  (8,280 possible actions)")
    print(f"Legal actions: {len(info['legal_actions'])} available")
    print(f"Side to move: {info['side_to_move']}")
    
    # Take a legal action
    action = info["legal_actions"][0]
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"\nAfter one move:")
    print(f"  Reward: {reward}, Terminated: {terminated}, Truncated: {truncated}")
    print(f"  Legal actions: {len(info['legal_actions'])}")
    print(f"  Side to move: {info['side_to_move']}")
    
except ImportError:
    print("⚠️ gymnasium not installed. Install with: pip install gymnasium")
    print("   The core environment works without it!")

---

## 9. Writing a Custom Agent

It's easy to create your own agent by subclassing `Agent`:

In [ ]:
from hybrid.agents.base import Agent
from hybrid.core.env import GameState
from hybrid.core.types import Move, PieceKind
from typing import List

class CaptureFirstAgent(Agent):
    """Agent that prefers capturing moves, then picks randomly."""
    name = "capture_first"
    
    def select_move(self, state: GameState, legal_moves: List[Move]) -> Move:
        # Separate captures from non-captures
        captures = []
        non_captures = []
        for mv in legal_moves:
            target = state.board.get(mv.tx, mv.ty)
            if target is not None:
                captures.append(mv)
            else:
                non_captures.append(mv)
        
        # Prefer captures (randomly among them), otherwise random non-capture
        if captures:
            return random.choice(captures)
        return random.choice(non_captures)

# Test it!
capture_agent = CaptureFirstAgent()
env = HybridChessEnv()
state = env.reset()
legal = env.legal_moves()
move = capture_agent.select_move(state, legal)
piece = state.board.get(move.fx, move.fy)
print(f"CaptureFirstAgent chose: {piece.kind.name} ({move.fx},{move.fy}) → ({move.tx},{move.ty})")

---

## 🎓 What's Next?

Now that you understand the game and environment, check out the next notebooks:

- **02: AI Agents & Search** — How AlphaBeta search works, comparing agent strengths
- **03: MCTS Explained** — Monte Carlo Tree Search from scratch, visualizing the search tree
- **04: AlphaZero Training** — Full training pipeline, from self-play to evaluation
- **05: Experiments** — Custom variants, reward shaping, architecture swaps

Or jump straight to training:

```bash
python -m hybrid train --iterations 10 --games 50 --simulations 50
```